## Basic Import

In [1]:
## Importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Importing wordcloud library for visualizing text data
! pip install wordcloud
from wordcloud import WordCloud

# Importing NLTK library for natural language processing tasks
import nltk
from nltk.corpus import stopwords

# Downloading NLTK Data
nltk.download('stopwords')
nltk.download('punkt')


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Atif\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Atif\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
# Read the csv fie into a pandas dataframe
df = pd.read_csv("spam.csv")

# Display the first few rows of the dataframe
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
# Drop the unnecessary columns
df.drop(columns = ['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)
# Display the first few rows of the dataframe after dropping unnecessary columns
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# Rename the columns for better understanding
df.rename(columns = {'v1':'target', 'v2':'text'}, inplace=True)
# Display the first few rows of the dataframe after renaming columns
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Data Preprocessing

In [5]:
from sklearn.preprocessing import LabelEncoder
# Initialize the LabelEncoder
encoder = LabelEncoder()
# Fit and transform the target column
df['target'] = encoder.fit_transform(df['target'])
# Display the first few rows of the dataframe after encoding the target column
df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
df.shape

(5572, 2)

In [ ]:
## Check the duplicate values in the dataframe
df.duplicated().sum()

np.int64(403)

In [9]:
len(df)

5572

In [10]:
# remove the duplicate values from the dataframe
df.drop_duplicates(keep='first', inplace=True)
len(df)

5169

## Feature Engineering

In [11]:
## Importing the PorterStemmer for stemming the text data
from nltk.stem.porter import PorterStemmer

## Importing string library for removing punctuation from the text data
import string

## Initializing the PorterStemmer
ps = PorterStemmer()

In [13]:
# Text Preprocessing Function
def transform_text(text):
    # Convert the text to lowercase
    text = text.lower()
    
    # Tokenize the text
    text = nltk.word_tokenize(text)
    
    # Remove the special characters
    y = []
    for i in text:
        if i.isalnum():
            y.append(i)

    text = y[:]
    y.clear()
    
    # Remove the stop words and punctuation
    stop_words = stopwords.words('english')
    for i in text:
        if i not in stop_words and i not in string.punctuation:
            y.append(i)
            
    text = y[:]
    y.clear()
    
    # Stemming the text
    for i in text:
        y.append(ps.stem(i))
    
    ## Joining the list of words back into a single string
    return " ".join(y)            
    
    
    

In [14]:
## Check the function with an example
transform_text("This is an example of text preprocessing using NLTK library in Python. It includes steps like tokenization, removing stop words, and stemming the text data.")

'exampl text preprocess use nltk librari python includ step like token remov stop word stem text data'

In [16]:
# transform_list = []
# for text in df['text']:
#     transform_list.append(transform_text(text))
# df['transformed_text'] = transform_list

df['transformed_text'] = df['text'].apply(transform_text)

df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [17]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
tfid= TfidfVectorizer(max_features=500)



In [18]:
X = tfid.fit_transform(df['transformed_text']).toarray()
X.shape

(5169, 500)

In [20]:
Y = df['target'].values
Y.shape

(5169,)

## Train Test Split

In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

## Model Training

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Create model objects
svc = SVC(kernel="sigmoid", gamma=1.0)
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth=5)
lrc = LogisticRegression(solver='liblinear', penalty='l1')
rfc = RandomForestClassifier(n_estimators=50, random_state=2)
abc = AdaBoostClassifier(n_estimators=50, random_state=2)
bc = BaggingClassifier(n_estimators=50, random_state=2)
etc = ExtraTreesClassifier(n_estimators=50, random_state=2)
gbdt = GradientBoostingClassifier(n_estimators=50, random_state=2)
xgb = XGBClassifier(n_estimators=50, random_state=2)

# Store in dictionary
clfs = {
    'SVC': svc,
    'KNN': knc,
    'NB': mnb,
    'DT': dtc,
    'LR': lrc,
    'RF': rfc,
    'Adaboost': abc,
    'Bgc': bc,
    'ETC': etc,
    'GBDT': gbdt,
    'xgb': xgb
}

## Model Evaluation

In [24]:
from sklearn.metrics import accuracy_score,precision_score
def train_classifier(model, X_train, Y_train, X_test, Y_test):
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(Y_test, y_pred)
    precision = precision_score(Y_test, y_pred)
    return accuracy, precision

accuracy_scores = []
precision_scores = []

for name,model in clfs.items():
    current_accuracy,current_precision = train_classifier(model, X_train, Y_train, X_test, Y_test)
    print("For ", name)
    print("Accuracy: ", current_accuracy)
    print("Precision: ", current_precision)
    accuracy_scores.append(current_accuracy)
    precision_scores.append(current_precision)

For  SVC
Accuracy:  0.9709864603481625
Precision:  0.952755905511811
For  KNN
Accuracy:  0.9294003868471954
Precision:  0.9736842105263158
For  NB
Accuracy:  0.9758220502901354
Precision:  0.9838709677419355
For  DT
Accuracy:  0.9313346228239845
Precision:  0.8363636363636363
For  LR
Accuracy:  0.9574468085106383
Precision:  0.904
For  RF
Accuracy:  0.9700193423597679
Precision:  0.9318181818181818
For  Adaboost
Accuracy:  0.9158607350096711
Precision:  0.7959183673469388
For  Bgc
Accuracy:  0.9584139264990329
Precision:  0.8642857142857143
For  ETC
Accuracy:  0.9758220502901354
Precision:  0.9477611940298507
For  GBDT
Accuracy:  0.9497098646034816
Precision:  0.9514563106796117
For  xgb
Accuracy:  0.9709864603481625
Precision:  0.96
